# Beer Distribution Game: short corrected Colab RL smoke

This notebook runs a deliberately short one-update, one-seed Tier-5 Y-wholesaler RL pipeline check. It pins the corrected trainer, uses T4-safe minibatches, and writes checkpoints to Google Drive. This is a pipeline smoke, not a publishable training result. Do not run validation until it completes successfully.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/ConstantinVictorBeatErtel/beer_distribution_RL.git"
REF = "5df7e1a"  # completion-only log-probability memory fix
REPO_DIR = pathlib.Path("/content/beer_distribution_RL")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", REF], check=True)
os.chdir(REPO_DIR)
print("checked out:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# Install the environment and training dependencies.
%pip install -q "verifiers==0.2.0" "transformers>=4.51,<6" "accelerate>=1.2" "peft>=0.14" "bitsandbytes>=0.45" safetensors
%pip install -q -e environments/beer_distribution_game --no-deps
subprocess.run([sys.executable, "-m", "py_compile", "scripts/train_colab_grpo_wholesaler.py"], check=True)
print("installation and compile check passed")

In [ ]:
# Preflight: require a Colab GPU, then confirm development-only task construction.
import shutil
nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("No NVIDIA GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, and rerun this notebook from the top.")
subprocess.run([nvidia_smi], check=False)
subprocess.run([
    sys.executable, "scripts/train_colab_grpo_wholesaler.py",
    "--dry-run", "--output-dir", "/content/beer_grpo_dryrun_fixed",
], check=True)

In [ ]:
# Run the short corrected RL pipeline smoke.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/beer_distribution_rl_artifacts/beer_wholesaler_qwen3_0p6b_grpo_quick_smoke"
RUN_ENV = {**os.environ, "PYTHONUNBUFFERED": "1"}
cmd = [
    sys.executable, "scripts/train_colab_grpo_wholesaler.py",
    "--updates", "1",
    "--group-size", "2",
    "--train-seeds", "0",
    "--train-minibatch", "1",
    "--inference-minibatch", "2",
    "--max-new-tokens", "16",
    "--prompt-max-tokens", "2048",
    "--output-dir", OUT,
]
print("Starting corrected smoke; output directory:", OUT)
result = subprocess.run(cmd, env=RUN_ENV, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Trainer failed with exit code {result.returncode}; see the traceback above.")
print("Quick smoke completed successfully.")

In [ ]:
# Held-out validation. Run only after the smoke cell reports success.
validation_out = OUT + "/validation_eval"
validation_cmd = [
    sys.executable, "scripts/train_colab_grpo_wholesaler.py",
    "--eval-only",
    "--adapter", OUT + "/adapter",
    "--eval-split", "validation",
    "--eval-seeds", "0", "1", "2", "3", "4",
    "--tier5-controls",
    "--output-dir", validation_out,
]
subprocess.run(validation_cmd, env=RUN_ENV, check=True)
print("Validation completed:", validation_out)